# 🏆 Proyecto Final 1: Predicción de Fuga de Clientes (Churn)
### Curso: **Machine Learning con Python** (IFCD093PO)
**Duración estimada:** 6 horas

---

## 🎯 Objetivo del Proyecto

¡Es hora de aplicar todo lo que has aprendido! En este proyecto, te enfrentarás a un problema de negocio real y muy común: **predecir qué clientes de una compañía de telecomunicaciones son propensos a cancelar su contrato (churn)**. Identificar a estos clientes de antemano permite a la empresa tomar acciones para retenerlos, como ofrecer descuentos o mejorar su servicio.

Seguiremos el flujo de trabajo completo de un proyecto de Machine Learning:

1.  **Análisis Exploratorio de Datos (EDA)**: Entender los datos y encontrar patrones iniciales.
2.  **Preprocesamiento y Preparación de Datos**: Limpiar y transformar los datos para los modelos.
3.  **Modelado**: Entrenar varios modelos de clasificación (Regresión Logística, Random Forest, XGBoost, etc.).
4.  **Evaluación y Selección del Modelo**: Comparar los modelos usando las métricas adecuadas (Accuracy, Precision, Recall, F1-Score, AUC) y elegir el mejor.
5.  **Conclusiones de Negocio**: Interpretar los resultados y proponer acciones basadas en ellos.

**Dataset:** Usaremos un dataset clásico de Churn de una compañía de telecomunicaciones, disponible públicamente (a menudo de Kaggle o la UCI). Contiene información demográfica de los clientes, los servicios que han contratado, su antigüedad y si finalmente se dieron de baja o no.

---

## 📚 Flujo de Trabajo

1. [Carga y Comprensión Inicial de los Datos](#1-carga)
2. [Análisis Exploratorio de Datos (EDA)](#2-eda)
   - [Análisis Univariado](#2.1-univariado)
   - [Análisis Bivariado](#2.2-bivariado)
   - [Correlaciones](#2.3-correlacion)
3. [Preprocesamiento de Datos](#3-prepro)
   - [Manejo de Valores Nulos](#3.1-nulos)
   - [Codificación de Variables Categóricas](#3.2-codificacion)
   - [Escalado de Características Numéricas](#3.3-escalado)
   - [División en Train/Test](#3.4-split)
4. [Modelado y Entrenamiento](#4-modelado)
   - [Modelo Base: Regresión Logística](#4.1-logistica)
   - [Modelo Potente: Random Forest](#4.2-randomforest)
   - [Modelo de Competición: XGBoost](#4.3-xgboost)
5. [Evaluación de Modelos](#5-evaluacion)
   - [Matriz de Confusión](#5.1-matriz)
   - [Métricas Clave y Curva ROC](#5.2-metricas)
   - [Selección del Mejor Modelo](#5.3-seleccion)
6. [Análisis de Importancia de Características](#6-features)
7. [Conclusiones y Recomendaciones de Negocio](#7-conclusiones)

---

## 📥 1. Carga y Comprensión Inicial de los Datos <a id='1-carga'></a>

El primer paso es cargar nuestro dataset y echar un vistazo rápido a su estructura, tipos de datos y estadísticas descriptivas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraciones de visualización
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# Cargar el dataset
# Usaremos un dataset conocido de Kaggle. Asegúrate de tener el archivo 'WA_Fn-UseC_-Telco-Customer-Churn.csv' en la misma carpeta o proporciona la ruta correcta.
try:
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
except FileNotFoundError:
    print("Archivo no encontrado. Por favor, descarga 'WA_Fn-UseC_-Telco-Customer-Churn.csv' de Kaggle y colócalo en la carpeta del proyecto.")
    # Podrías añadir aquí una descarga automática si lo deseas
    # import requests
    # url = 'URL_AL_RAW_CSV'
    # r = requests.get(url)
    # with open('WA_Fn-UseC_-Telco-Customer-Churn.csv', 'wb') as f:
    #     f.write(r.content)
    # df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

if 'df' in locals():
    print("Dataset cargado exitosamente.")
    print("Dimensiones del dataset:", df.shape)
    print("\nPrimeras 5 filas:")
    display(df.head())
    
    print("\nInformación del dataset:")
    df.info()
    
    print("\nEstadísticas descriptivas:")
    display(df.describe())

**Observaciones Iniciales:**
- `customerID` es un identificador único, probablemente no será útil para el modelo.
- `TotalCharges` es de tipo `object`, pero debería ser numérico. Hay que investigar por qué.
- Hay varias columnas categóricas con valores 'Yes', 'No', 'No phone service', etc. que necesitarán ser codificadas.
- La variable objetivo es `Churn` ('Yes' o 'No').

---

## 📊 2. Análisis Exploratorio de Datos (EDA) <a id='2-eda'></a>

En esta fase, visualizaremos los datos para entender la distribución de las variables y su relación con la variable objetivo `Churn`.

In [ ]:
if 'df' in locals():
    # Corregir 'TotalCharges'
    # Algunos valores pueden ser espacios en blanco, los convertimos a NaN y luego a float
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    
    # Visualizar la distribución de la variable objetivo
    plt.figure(figsize=(6, 4))
    sns.countplot(x='Churn', data=df)
    plt.title('Distribución de Churn')
    plt.show()
    
    churn_rate = df['Churn'].value_counts(normalize=True)
    print(f"Tasa de Churn:\n{churn_rate * 100}")

**Observación:** Vemos un desbalance en las clases. Aproximadamente el 26.5% de los clientes en el dataset han cancelado el servicio. Esto es importante tenerlo en cuenta para la evaluación del modelo (la accuracy por sí sola no será una buena métrica).

In [ ]:
if 'df' in locals():
    # Análisis de variables categóricas vs Churn
    categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                            'PhoneService', 'MultipleLines', 'InternetService', 
                            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                            'TechSupport', 'StreamingTV', 'StreamingMovies', 
                            'Contract', 'PaperlessBilling', 'PaymentMethod']

    fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 18))
    axes = axes.flatten()

    for i, col in enumerate(categorical_features):
        sns.countplot(x=col, hue='Churn', data=df, ax=axes[i])
        axes[i].set_title(f'{col} vs Churn')
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

**Observaciones del EDA (Categóricas):**
- Los clientes con contratos `Month-to-month` tienen una tasa de churn mucho más alta.
- Los clientes sin `OnlineSecurity` o `TechSupport` parecen irse más.
- Los clientes con `Fiber optic` como servicio de internet tienen más churn que los de DSL.
- El método de pago `Electronic check` está asociado con un mayor churn.

In [ ]:
if 'df' in locals():
    # Análisis de variables numéricas vs Churn
    numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5))

    for i, col in enumerate(numerical_features):
        sns.histplot(data=df, x=col, hue='Churn', multiple='stack', ax=axes[i])
        axes[i].set_title(f'{col} vs Churn')

    plt.tight_layout()
    plt.show()

**Observaciones del EDA (Numéricas):**
- `tenure` (antigüedad): Los clientes nuevos (baja antigüedad) tienen una probabilidad mucho mayor de irse.
- `MonthlyCharges`: Los clientes con cargos mensuales más altos tienden a hacer churn más a menudo.

---

## ⚙️ 3. Preprocesamiento de Datos <a id='3-prepro'></a>

Ahora prepararemos los datos para que puedan ser utilizados por los algoritmos de Machine Learning.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

if 'df' in locals():
    # Eliminar customerID
    df_processed = df.drop('customerID', axis=1)

    # Convertir Churn a numérico
    df_processed['Churn'] = df_processed['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

    # Separar X e y
    X = df_processed.drop('Churn', axis=1)
    y = df_processed['Churn']

    # Identificar columnas numéricas y categóricas
    # (Excluimos las que ya hemos procesado o vamos a dropear)
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    # Crear pipelines de preprocesamiento, aqui incluimos imputación y escalado/one-hot encoding
    # La función Pipeline nos permite encadenar varios pasos de transformación
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')), # Usamos mediana para ser robustos a outliers
        ('scaler', StandardScaler())]) # Escalado estándar

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')), # Imputar con la moda
        ('onehot', OneHotEncoder(handle_unknown='ignore'))]) # One-hot encoding

    # Crear el preprocesador con ColumnTransformer
    preprocessor = ColumnTransformer( # Esto nos permite aplicar diferentes transformaciones a diferentes columnas
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)])

    # Dividir los datos ANTES de aplicar el preprocesador para evitar data leakage es decir, que la información del test set influya en el train set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("Preprocesamiento definido. Datos listos para ser transformados dentro de un pipeline de modelo.")
    print("Tamaño de Train set:", X_train.shape)
    print("Tamaño de Test set:", X_test.shape)

---

## 🤖 4. Modelado y Entrenamiento <a id='4-modelado'></a>

Crearemos pipelines que combinen el preprocesamiento con cada modelo. Esto asegura que los datos de test se transformen usando la información aprendida solo en los datos de train.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.metrics import RocCurveDisplay

if 'df' in locals(): # Asegurarse de que el preprocesador y los datos están definidos. locals() sirve para ver las variables locales
    # --- 1. Regresión Logística ---
    pipeline_lr = Pipeline(steps=[('preprocessor', preprocessor), # Encadenar preprocesador y clasificador
                                  ('classifier', LogisticRegression(random_state=42, max_iter=1000))]) # Clasificador
    pipeline_lr.fit(X_train, y_train)
    y_pred_lr = pipeline_lr.predict(X_test)
    print("--- Resultados Regresión Logística ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, pipeline_lr.predict_proba(X_test)[:, 1]):.4f}")

    # --- 2. Random Forest ---
    pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('classifier', RandomForestClassifier(random_state=42))])
    pipeline_rf.fit(X_train, y_train)
    y_pred_rf = pipeline_rf.predict(X_test)
    print("\n--- Resultados Random Forest ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, pipeline_rf.predict_proba(X_test)[:, 1]):.4f}")

    # --- 3. XGBoost ---
    pipeline_xgb = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))])
    pipeline_xgb.fit(X_train, y_train)
    y_pred_xgb = pipeline_xgb.predict(X_test)
    print("\n--- Resultados XGBoost ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, pipeline_xgb.predict_proba(X_test)[:, 1]):.4f}")

---

## 📈 5. Evaluación de Modelos <a id='5-evaluacion'></a>

Ahora compararemos los modelos de forma más detallada, centrándonos en las métricas que importan para un problema de churn.

In [ ]:
from sklearn.metrics import RocCurveDisplay

if 'df' in locals():
    print("--- Reporte de Clasificación: Regresión Logística ---")
    print(classification_report(y_test, y_pred_lr))

    print("\n--- Reporte de Clasificación: Random Forest ---")
    print(classification_report(y_test, y_pred_rf))

    print("\n--- Reporte de Clasificación: XGBoost ---")
    print(classification_report(y_test, y_pred_xgb))
    
    # Curva ROC
    fig, ax = plt.subplots(figsize=(10, 8))
    RocCurveDisplay.from_estimator(pipeline_lr, X_test, y_test, name='Logistic Regression', ax=ax)
    RocCurveDisplay.from_estimator(pipeline_rf, X_test, y_test, name='Random Forest', ax=ax)
    RocCurveDisplay.from_estimator(pipeline_xgb, X_test, y_test, name='XGBoost', ax=ax)
    plt.title('Curva ROC - Comparación de Modelos')
    plt.show()

### 5.3 Selección del Mejor Modelo <a id='5.3-seleccion'></a>

**Análisis:**
- **ROC AUC**: La Regresión Logística tiene el mejor rendimiento según esta métrica, que mide la capacidad del modelo para distinguir entre clases.
- **Precision vs. Recall (para la clase 1 - Churn):**
  - **Recall (Sensibilidad)**: ¿Qué proporción de los clientes que realmente hicieron churn fuimos capaces de identificar? La Regresión Logística tiene un recall de 0.55, mientras que Random Forest tiene 0.49 y XGBoost 0.52. 
  - **Precision**: De todos los clientes que predijimos que harían churn, ¿cuántos lo hicieron realmente? La Regresión Logística tiene una precisión de 0.66, RF 0.64 y XGBoost 0.63.

**Conclusión de la selección:**
Para un problema de churn, a menudo es más costoso no identificar a un cliente que se va a ir (Falso Negativo) que contactar a un cliente que no se iba a ir (Falso Positivo). Por lo tanto, un **recall** más alto suele ser preferible, incluso a costa de una menor precisión.

La **Regresión Logística** ofrece el mejor equilibrio general, con el ROC AUC más alto y el mejor recall para la clase positiva. Aunque los otros modelos son más complejos, en este caso, el modelo más simple generaliza mejor. Podríamos mejorar los modelos de ensemble con un ajuste de hiperparámetros, pero el modelo lineal ya es un punto de partida muy sólido.

---

## 🔬 6. Análisis de Importancia de Características <a id='6-features'></a>

¿Qué factores influyen más en la decisión de un cliente de irse? Podemos extraer los coeficientes de nuestro modelo de Regresión Logística para entenderlo.

In [ ]:
if 'df' in locals():
    # Obtener los nombres de las características después del OneHotEncoding
    feature_names = pipeline_lr.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
    all_feature_names = np.concatenate([numerical_features, feature_names])

    # Obtener los coeficientes del modelo
    coefs = pipeline_lr.named_steps['classifier'].coef_[0]

    # Crear un DataFrame para visualizarlos
    coef_df = pd.DataFrame({'feature': all_feature_names, 'coefficient': coefs})
    coef_df['abs_coefficient'] = np.abs(coef_df['coefficient'])
    coef_df = coef_df.sort_values('abs_coefficient', ascending=False)

    plt.figure(figsize=(10, 12))
    sns.barplot(x='coefficient', y='feature', data=coef_df.head(20))
    plt.title('Top 20 Características más Influyentes (Regresión Logística)')
    plt.show()

---

## 🏁 7. Conclusiones y Recomendaciones de Negocio <a id='7-conclusiones'></a>

Basado en nuestro análisis y el modelo final, podemos extraer conclusiones valiosas para el negocio:

1.  **Modelo Predictivo**: Hemos desarrollado un modelo de Regresión Logística capaz de predecir la fuga de clientes con un ROC AUC de ~0.84. Este modelo puede ser usado para generar una lista de clientes "en riesgo" cada mes.

2.  **Factores de Riesgo Clave (Coeficientes Positivos Altos)**:
    - **Contrato Mes a Mes**: Es el predictor más fuerte de churn. Los clientes sin un compromiso a largo plazo son muy volátiles.
    - **Internet de Fibra Óptica**: Aunque es un servicio premium, está asociado con un mayor churn. Esto podría indicar problemas de precio, fiabilidad o competencia en esa área.
    - **Facturación sin Papel y Pago con Cheque Electrónico**: Pueden ser indicativos de un perfil de cliente menos "atado" a la compañía.

3.  **Factores de Retención Clave (Coeficientes Negativos Altos)**:
    - **Antigüedad (tenure)**: Cuanto más tiempo lleva un cliente, menos probable es que se vaya. La lealtad es un factor protector enorme.
    - **Contratos a Largo Plazo (Uno o Dos Años)**: Son el factor más importante para retener a un cliente.
    - **Servicios de Soporte**: Tener `OnlineSecurity` y `TechSupport` reduce significativamente el churn.

### Recomendaciones de Negocio:

- **Acción Inmediata**: Utilizar el modelo para identificar a los clientes con mayor probabilidad de churn y dirigirles campañas de retención proactivas (ej. una oferta de descuento para pasar a un contrato anual).
- **Estrategia a Largo Plazo**:
  - **Incentivar Contratos Anuales**: Crear ofertas atractivas para que los clientes de mes a mes se cambien a contratos de 1 o 2 años.
  - **Investigar el Churn en Fibra Óptica**: ¿Es un problema de precio? ¿De calidad del servicio? Se necesita un análisis más profundo en este segmento.
  - **Promocionar Servicios de Soporte**: Ofrecer `OnlineSecurity` y `TechSupport` de forma gratuita o con descuento durante los primeros meses puede aumentar la "pegajosidad" del cliente.